# S6E9 | 0.94644 LB vs OOF: Slot-Saving Playbook

**Five real submissions. Three public OOF sources. One rule: do not spend a leaderboard slot until the gain survives validation.**

Public ROC-AUC differences in this competition are tiny enough that a blend can look better on the leaderboard while being worse under honest OOF validation. I learned that the expensive way, so this notebook turns those experiments into a reusable workflow.

### TL;DR

- A strong raw LightGBM reproduced at **OOF 0.946264** and **public LB 0.94633**.
- An OOF-first tree blend moved to about **0.946361 OOF** and **0.94640 public LB**.
- A public reference reached **0.94644**, but that does **not** mean every ingredient in it is robust.
- Adding a genuinely different RealMLP gives a small but repeatable OOF gain; the nested check below lands around **0.946369**.
- A tempting `income % 1000` target-encoding idea actually made OOF worse.

The goal here is not to publish another giant ensemble. It is to show **which checks are worth doing before you burn a daily submission**.

## 1. What five live submissions taught me

These are public-leaderboard measurements from my account during the same experiment cycle. The 0.94644 line is a **public reference blend/output**, not a claim that I invented all of its component models.

| experiment | local OOF | public LB | lesson |
|---|---:|---:|---|
| raw LightGBM reproduction | 0.946264 | 0.94633 | a strong single model is the best anchor |
| OOF-first rank blend | 0.946361 | 0.94640 | OOF diversity translated to LB |
| Zoom Zoom public reference | - | 0.94643 | strong public anchor |
| reference + 5.5% Transformer | - | 0.94642 | decorrelation alone is not enough |
| public 0.94644 reference blend | - | **0.94644** | excellent public score, but treat LB-only weights carefully |

That last point matters: **public score and model quality are related, but they are not the same object.** The rest of this notebook uses OOF labels to decide what survives.

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

TARGET = "Will_Buy_EV"
ID = "id"


def find_one(filename, contains=None):
    search_root = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path.cwd()
    hits = sorted(glob.glob(str(search_root / "**" / filename), recursive=True))
    if contains is not None:
        filtered = [p for p in hits if contains.lower() in p.lower()]
        if filtered:
            hits = filtered
    if len(hits) != 1:
        raise FileNotFoundError(f"Expected one {filename!r}; found {len(hits)}: {hits[:8]}")
    return Path(hits[0])


def rank01(values):
    values = np.asarray(values, dtype=float)
    return (rankdata(values, method="average") - 0.5) / len(values)


def auc(y, p):
    return float(roc_auc_score(y, p))

train_path = find_one("train.csv", "playground-series-s6e9")
test_path = find_one("test.csv", "playground-series-s6e9")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
y = train[TARGET].map({"No": 0, "Yes": 1}).to_numpy(np.int8)

print("train:", train.shape, "test:", test.shape)
print("positive rate:", f"{y.mean():.3%}")

## 2. Load three public OOF/test pairs

I deliberately keep this small:

1. **Raw LightGBM** - strong tree anchor, OOF around 0.946264.
2. **Naji Power-Two OOF blend** - a second strong tree view with different feature engineering.
3. **RealMLP** - weaker alone, but much more architecturally different.

All three sources publish OOF predictions, so we can evaluate them against the competition train labels instead of guessing from the public leaderboard.

In [ ]:
raw_oof_path = find_one("oof_hybrid.npy")
naji_oof_path = find_one("01_blend_oof.csv")
real_oof_path = find_one("oof_preds.csv")

raw_test_path = raw_oof_path.with_name("test_hybrid.npy")
naji_test_path = naji_oof_path.with_name("01_submission.csv")
real_test_path = real_oof_path.with_name("submission.csv")

raw_oof = np.load(raw_oof_path)
raw_test = np.load(raw_test_path)

naji_oof_df = pd.read_csv(naji_oof_path)
naji_test_df = pd.read_csv(naji_test_path)
real_oof_df = pd.read_csv(real_oof_path)
real_test_df = pd.read_csv(real_test_path)

assert len(raw_oof) == len(train) and len(raw_test) == len(test)
assert naji_oof_df[ID].equals(train[ID])
assert real_oof_df[ID].equals(train[ID])
assert naji_test_df[ID].equals(test[ID])
assert real_test_df[ID].equals(test[ID])

for frame in (naji_oof_df, naji_test_df, real_oof_df, real_test_df):
    assert np.isfinite(frame[TARGET]).all()

OOF = {
    "raw_lgbm": rank01(raw_oof),
    "naji_power_two": rank01(naji_oof_df[TARGET]),
    "realmlp": rank01(real_oof_df[TARGET]),
}
TEST = {
    "raw_lgbm": rank01(raw_test),
    "naji_power_two": rank01(naji_test_df[TARGET]),
    "realmlp": rank01(real_test_df[TARGET]),
}

score_table = pd.DataFrame({
    "OOF_AUC": {name: auc(y, pred) for name, pred in OOF.items()},
    "Spearman_vs_raw": {
        name: spearmanr(OOF["raw_lgbm"], pred).statistic for name, pred in OOF.items()
    },
}).sort_values("OOF_AUC", ascending=False)
display(score_table.style.format("{:.9f}"))

In [ ]:
corr = pd.DataFrame(OOF).corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(corr.to_numpy(), vmin=0.99, vmax=1.0, cmap="viridis")
ax.set_xticks(range(len(corr)), corr.columns, rotation=25, ha="right")
ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.5f}", ha="center", va="center", color="white")
ax.set_title("OOF Spearman correlation")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### The important pattern

RealMLP is weaker by itself, but it disagrees with the two tree systems more often. That is exactly the kind of model that *can* help a saturated ensemble. The next question is whether the gain survives a validation scheme that does not use the same rows to choose and judge the weight.

## 3. Build a simple tree core, then nest only the neural weight

I lock the raw/Naji tree ratio near a broad OOF plateau instead of fitting dozens of free weights. Then I choose the RealMLP weight on four meta-folds and evaluate it on the held-out fifth fold.

This is intentionally boring. Boring is good when leaderboard differences live in the fifth decimal.

In [ ]:
RAW_SHARE = 0.612
core_oof = RAW_SHARE * OOF["raw_lgbm"] + (1 - RAW_SHARE) * OOF["naji_power_two"]
core_test = RAW_SHARE * TEST["raw_lgbm"] + (1 - RAW_SHARE) * TEST["naji_power_two"]
core_auc = auc(y, core_oof)
print(f"locked tree core OOF AUC: {core_auc:.9f}")

raw_grid = np.arange(0.50, 0.701, 0.01)
core_sweep = pd.DataFrame({
    "raw_share": raw_grid,
    "OOF_AUC": [
        auc(y, w * OOF["raw_lgbm"] + (1 - w) * OOF["naji_power_two"])
        for w in raw_grid
    ],
})
display(core_sweep.sort_values("OOF_AUC", ascending=False).head(7).style.format({"raw_share": "{:.2f}", "OOF_AUC": "{:.9f}"}))

In [ ]:
REAL_GRID = np.arange(0.05, 0.181, 0.01)
meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=20260912)
nested_pred = np.zeros(len(y), dtype=float)
rows = []

for fold, (tr, va) in enumerate(meta_cv.split(core_oof, y)):
    train_scores = [
        auc(y[tr], (1 - w) * core_oof[tr] + w * OOF["realmlp"][tr])
        for w in REAL_GRID
    ]
    best_w = float(REAL_GRID[int(np.argmax(train_scores))])
    fold_pred = (1 - best_w) * core_oof[va] + best_w * OOF["realmlp"][va]
    nested_pred[va] = fold_pred
    rows.append({
        "fold": fold,
        "selected_realmlp_weight": best_w,
        "core_auc": auc(y[va], core_oof[va]),
        "candidate_auc": auc(y[va], fold_pred),
        "gain": auc(y[va], fold_pred) - auc(y[va], core_oof[va]),
    })

nested = pd.DataFrame(rows)
nested_auc = auc(y, nested_pred)
print(f"core OOF AUC:          {core_auc:.9f}")
print(f"nested candidate AUC:  {nested_auc:.9f}")
print(f"nested gain:           {nested_auc - core_auc:+.9f}")
display(nested.style.format({
    "selected_realmlp_weight": "{:.2f}",
    "core_auc": "{:.9f}",
    "candidate_auc": "{:.9f}",
    "gain": "{:+.9f}",
}))

In [ ]:
full_sweep = pd.DataFrame({
    "realmlp_weight": REAL_GRID,
    "OOF_AUC": [
        auc(y, (1 - w) * core_oof + w * OOF["realmlp"])
        for w in REAL_GRID
    ],
})

LOCKED_REAL_WEIGHT = 0.11
final_oof = (1 - LOCKED_REAL_WEIGHT) * core_oof + LOCKED_REAL_WEIGHT * OOF["realmlp"]
final_auc = auc(y, final_oof)

fig, ax = plt.subplots(figsize=(8.0, 4.6))
ax.plot(full_sweep["realmlp_weight"], full_sweep["OOF_AUC"], marker="o")
ax.axvline(LOCKED_REAL_WEIGHT, linestyle="--", label="locked 11%")
ax.set_xlabel("RealMLP weight")
ax.set_ylabel("OOF ROC-AUC")
ax.set_title("A broad local plateau is safer than a sharp optimum")
ax.legend()
plt.tight_layout()
plt.show()

print(f"final locked OOF AUC: {final_auc:.9f}")
print("final effective weights:", {
    "raw_lgbm": round((1 - LOCKED_REAL_WEIGHT) * RAW_SHARE, 6),
    "naji_power_two": round((1 - LOCKED_REAL_WEIGHT) * (1 - RAW_SHARE), 6),
    "realmlp": LOCKED_REAL_WEIGHT,
})

## 4. Three tempting ideas I rejected

Negative results are often more useful than another leaderboard screenshot. These were all tested in the same project and **did not pass the gate**:

| candidate | result | decision |
|---|---:|---|
| add fold-safe `income % 1000` / recipe-cell target encodings to the raw LGBM | 0.94626423 -> **0.94624742 OOF** | reject |
| add a competition-only LGBM to the already-extended OOF blend | nested gain **-0.00000283** | reject |
| add 5.5% Transformer to a 0.94643 public reference | **0.94642 public LB** | reject |

The recurring lesson is simple: **a plausible feature story, a diverse architecture, or a strong standalone score is not enough. The candidate has to improve the ensemble you actually have.**

## 5. Write a submission with hard QA gates

The notebook finishes by writing a rank-based submission from the locked three-source blend. I do not claim a public score for this file until Kaggle evaluates it; the point is that its weighting passed the OOF checks above.

In [ ]:
final_test = rank01(
    (1 - LOCKED_REAL_WEIGHT) * core_test
    + LOCKED_REAL_WEIGHT * TEST["realmlp"]
)

submission = pd.DataFrame({ID: test[ID].to_numpy(), TARGET: final_test})
assert len(submission) == len(test)
assert submission[ID].is_unique
assert submission[ID].equals(test[ID])
assert np.isfinite(submission[TARGET]).all()
assert submission[TARGET].between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print("wrote submission.csv")
print({
    "rows": len(submission),
    "min": float(submission[TARGET].min()),
    "max": float(submission[TARGET].max()),
    "OOF_AUC_locked_blend": final_auc,
    "nested_AUC_weight_selection": nested_auc,
})
display(submission.head())

## 6. My 30-second checklist before using a submission slot

Use this on any tabular ROC-AUC competition:

1. **Is the candidate better OOF than the current ensemble, not just better alone?**
2. **Does the weight survive a held-out or nested check?**
3. **Is the candidate different enough to change ranking errors?** Check Spearman, not just model family names.
4. **Is there a plateau?** A broad stable region is safer than a one-point optimum.
5. **Did an ablation beat the exact previous pipeline?** Do not compare across drifting CV setups.
6. **Are IDs, order, row count, finite values and target bounds verified?**
7. **Can you explain the gain without saying only “LB went up”?** If not, keep investigating.

This checklist saved me more submission slots than another round of micro-weight tuning would have.

## Credits and reproducibility

This notebook intentionally builds on public community work and keeps the modeling credit visible:

- **megayak / Artificial Idiocy Lab** - [One LightGBM From Raw Data, CV 0.9463](https://www.kaggle.com/code/megayak/s6e9-one-lightgbm-from-raw-data-cv-0-9463)
- **Naji** - [S6E9 OOF dataset](https://www.kaggle.com/datasets/najiama/s6e9-oof) and Power-Two blend work
- **yekenot** - [RealMLP - PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch)
- **Talha Tursun** - [Daily Rank-Average Ensemble](https://www.kaggle.com/code/talhatursun/s6e9-daily-rank-average-ensemble)
- **jazivxt** - Zoom Zoom reference family
- **Lamhuy8904** - Transformer + GBDT diversity experiment

What is new here is the **OOF-first screening workflow, the nested RealMLP weight check, the controlled negative-result log, and the slot-saving decision checklist** tied to my five live submissions.

If you fork this, I would be especially interested in counterexamples: a candidate that passes these gates but still fails badly on the public or private leaderboard. Feedback and reproducible comparisons are welcome.